# 3) Merge SATCAT and GP


In [ ]:
# Inputs: processed CSVs | Process: load latest SATCAT/GP | Outputs: dfs
import pandas as pd
from pathlib import Path

satcat_path = sorted(Path("../data/processed").glob("satcat_*.csv"))[-1]
gp_path = sorted(Path("../data/processed").glob("gp_*.csv"))[-1]
satcat_df = pd.read_csv(satcat_path)
gp_df = pd.read_csv(gp_path)
print(satcat_df.shape, gp_df.shape)


pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', None)

satcat_df.head(10)
gp_df.head(10)

In [ ]:
# Inputs: satcat_df,gp_df | Process: left-join on NORAD key | Outputs: satnav_df
satnav_df = pd.merge(satcat_df, gp_df, left_on="NORAD_CAT_ID", right_on="NORAD_CAT_ID", how="left")
print(satnav_df.shape)
satnav_df.head()


In [41]:
# Inputs: satnav_df | Process: basic DQ | Outputs: info + nulls
print(satnav_df.info())
print(satnav_df.isnull().sum())


def show_freeze_header(df, height=400):
    html = df.to_html(index=False)
    return HTML(f"""
    <style>
    .scrollbox {{ max-height:{height}px; overflow-y:auto; }}
    .scrollbox thead th {{ position: sticky; top: 0; background: white; z-index: 2; }}
    </style>
    <div class="scrollbox">{html}</div>
    """)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 65088 entries, 0 to 65087
Data columns (total 63 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   intldes              65088 non-null  object 
 1   norad_cat_id         65088 non-null  int64  
 2   object_type_x        65088 non-null  object 
 3   satname              65088 non-null  object 
 4   country              65088 non-null  object 
 5   launch               65088 non-null  object 
 6   site_x               65088 non-null  object 
 7   decay                33937 non-null  object 
 8   period_x             64101 non-null  float64
 9   inclination_x        64101 non-null  float64
 10  apogee               64101 non-null  float64
 11  perigee              64101 non-null  float64
 12  comment_x            960 non-null    object 
 13  commentcode          3439 non-null   float64
 14  rcsvalue             65088 non-null  int64  
 15  rcs_size_x           54605 non-null 

In [ ]:
# Inputs: satnav_df | Process: save merged latest | Outputs: CSV
from pathlib import Path
out_path = Path("../data/processed/satnav_latest.csv")
satnav_df.to_csv(out_path, index=False)
print("Saved:", out_path)


In [ ]:
# Inputs: satnav_df | Process: set pandas display options and inspect tail | Outputs: formatted tail transpose
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
satnav_df.tail().T

In [ ]:
# Inputs: satnav_df | Process: drop verbose or redundant columns | Outputs: reduced satnav_df
keep_cols = ['INTLDES', 'NORAD_CAT_ID', 'SATNAME', 'COUNTRY', 'LAUNCH', '']
drop_cols = ['OBJECT_ID_x', 'OBJECT_ID_y', 'OBJECT_TYPE_x', 'OBJECT_TYPE_y', 'OBJECT_NAME_x', 'OBJECT_NAME_y', 'COMMENT_y', 'COMMENTCODE', 'RCSVALUE', 'FILE_x', 'FILE_y', 'CURRENT', 'OBJECT_NUMBER', 'CCSDS_OMM_VERS', 'ORIGINATOR', 'SITE_x', 'SITE_y', 'APOGEE', 'PERIGEE', 'PERIOD_x', 'PERIOD_y', 'INCLINATION_x', 'INCLINATION_y', 'ECCENTRICITY_x', 'APOAPSIS', 'ECCENTRICITY_y', 'PERIAPSIS', 'EPHEMERIS_TYPE', 'RCS_SIZE_x', 'RCS_SIZE_y', 'COUNTRY_CODE', 'COUNTRY']
satnav_df = satnav_df.drop(columns=drop_cols, errors="ignore")

In [ ]:
# Inputs: satnav_df | Process: consolidate country fields (prefer COUNTRY over COUNTRY_CODE) | Outputs: satnav_df['country']
satnav_df['country'] = satnav_df['COUNTRY'].combine_first(satnav_df['COUNTRY_CODE'])


In [ ]:
# Inputs: satnav_df | Process: compare COUNTRY_CODE vs COUNTRY for non-null mismatches | Outputs: mismatch count + sample rows
x = 'COUNTRY_CODE'
y = 'COUNTRY'

valid = satnav_df[x].notna() & satnav_df[y].notna()
diff_rows = satnav_df[valid & (satnav_df[x] != satnav_df[y])]
print(len(diff_rows))
print(satnav_df[x].notna().sum())
print(satnav_df[y].notna().sum())
print(diff_rows[['COUNTRY_CODE', 'COUNTRY']].head())

In [ ]:
# Inputs: satnav_df | Process: detect duplicates by INTLDES key (here SATNAME used as proxy) | Outputs: counts + sample
# All rows where INTLDES appears more than once
duplicates_df = satnav_df[satnav_df.duplicated('SATNAME', keep=False)].sort_values('SATNAME')

# Counts per INTLDES and the subset that are duplicated
intldes_counts = satnav_df['SATNAME'].value_counts(dropna=False)
duplicated_counts = intldes_counts[intldes_counts > 1]

print("Number of INTLDES with duplicates:", duplicated_counts.shape[0])
print(duplicated_counts.head())

# Optional: inspect duplicate rows
duplicates_df.head(20)

In [ ]:
# Inputs: diff_rows | Process: inspect slice of mismatches | Outputs: transposed view
diff_rows[10:20].T

In [ ]:
# Inputs: satnav_df | Process: convert NaN to None for serialization/DB compatibility | Outputs: updated satnav_df
satnav_df = satnav_df.where(pd.notnull(satnav_df), None)

In [ ]:
# Inputs: satnav_df | Process: inspect unique values of selected categorical columns | Outputs: unique lists
check_unique = ['CLASSIFICATION_TYPE']
for col in check_unique:
    print(col)
    print(satnav_df[col].unique())
    print("-"*100)

In [ ]:
satnav_df.tail().T

In [ ]:
satnav_df.columns = satnav_df.columns.str.strip().str.lower()

In [ ]:
# Inputs: gp_df | Process: add length column and show duplicates | Outputs: df with len_OBJECT
df1 = satcat_df[['INTLDES', 'SATNAME']]
df2 = gp_df[['OBJECT_ID', 'OBJECT_NAME']].copy()

df2['len_OBJECT'] = df2['OBJECT_ID'].astype(str).str.len()

df2_dups = df2[df2.duplicated('OBJECT_ID', keep=False)]
df2_dups[['OBJECT_ID', 'OBJECT_NAME', 'len_OBJECT']].head(20)